In [1]:
!pip install xgboost shap -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(r"D:\G5HackatonAI\data\processed\cleaned_dataset.csv")
df.head()

,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,Termd,PositionID,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,0,0,1,1,5,4,0,62506,0,19,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,1/17/2019,0,1
1,1,1,1,5,3,3,0,104437,1,27,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2/24/2016,0,17
2,1,1,0,5,5,3,0,64955,1,20,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,5/15/2012,0,3
3,1,1,0,1,5,3,0,64991,0,19,...,Elijiah Gray,16.0,Indeed,Fully Meets,4.84,5,0,1/3/2019,0,15
4,0,2,0,5,5,3,0,50825,1,19,...,Webster Butler,39.0,Google Search,Fully Meets,5.00,4,0,2/1/2016,0,2


In [3]:

!pip install vaderSentiment -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer # pip install vaderSentiment

print("=" * 65)
print(" HR GUARDIAN AI - Data Loading")
print("=" * 65)

df = pd.read_csv("D:\G5HackatonAI\data\processed\cleaned_dataset.csv")

print(f" Dataset Loaded: {df.shape[0]} employees, {df.shape[1]} features")
print(f" Target Distribution (Termd):")
print(f"   → Active (0): {(df['Termd'] == 0).sum()} employees")
print(f"   → Terminated (1): {(df['Termd'] == 1).sum()} employees")

 HR GUARDIAN AI - Data Loading
 Dataset Loaded: 311 employees, 34 features
 Target Distribution (Termd):
   → Active (0): 207 employees
   → Terminated (1): 104 employees


In [4]:

print("\n" + "=" * 65)
print(" STEP 2: Feature Engineering & NLP")
print("=" * 65)

reference_date = pd.to_datetime('2018-12-31')
df['DateofHire'] = pd.to_datetime(df['DateofHire'], errors='coerce')
df['TenureYears'] = (reference_date - df['DateofHire']).dt.days / 365.25
df['TenureYears'] = df['TenureYears'].fillna(0) 

print("Feature Created: 'TenureYears' (derived from DateofHire)")

def generate_feedback(satisfaction):
    if satisfaction >= 4: return "I am very happy here, great team and good management."
    elif satisfaction == 3: return "Things are okay, but I feel neutral about my career progression."
    else: return "I am frustrated with my salary and lack of opportunities."

df['Simulated_Review_Notes'] = df['EmpSatisfaction'].apply(generate_feedback)

analyzer = SentimentIntensityAnalyzer()
df['NLP_Sentiment_Score'] = df['Simulated_Review_Notes'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
print(" NLP Applied: Sentiment Analysis extracted from Simulated Review Notes")

IDENTIFIERS = ["DOB", "State", "Zip", "DateofHire", "LastPerformanceReview_Date", "Simulated_Review_Notes"]
DATA_LEAKAGE = ["DateofTermination", "TermReason", "EmploymentStatus", "EmpStatusID"]

df_clean = df.drop(columns=IDENTIFIERS + DATA_LEAKAGE, errors="ignore")
print(f" Dropped {len(IDENTIFIERS + DATA_LEAKAGE)} columns (Leakage & GDPR Compliance)")

CATEGORICAL_COLS = ["Sex", "MaritalDesc", "CitizenDesc", "HispanicLatino", "RaceDesc",
                    "Department", "Position", "ManagerName", "RecruitmentSource", "PerformanceScore"]

le = LabelEncoder()
for col in CATEGORICAL_COLS:
    if col in df_clean.columns:
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))

print(" Categorical features encoded to numeric.")


 STEP 2: Feature Engineering & NLP
Feature Created: 'TenureYears' (derived from DateofHire)
 NLP Applied: Sentiment Analysis extracted from Simulated Review Notes
 Dropped 10 columns (Leakage & GDPR Compliance)
 Categorical features encoded to numeric.


In [5]:
print("\n" + "=" * 65)
print(" STEP 2.5: HR Business Insights (EDA)")
print("=" * 65)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Global HR Insights - Turnover Drivers", fontsize=16, fontweight="bold")

dept_turnover = df.groupby('Department')['Termd'].mean().sort_values(ascending=False)
sns.barplot(x=dept_turnover.values, y=dept_turnover.index, ax=axes[0], palette="viridis")
axes[0].set_title("Turnover Rate by Department")
axes[0].set_xlabel("Turnover Rate (%)")
axes[0].set_ylabel("")

sns.boxplot(data=df, x='Termd', y='Salary', ax=axes[1], palette="Set2")
axes[1].set_title("Salary Distribution: Active (0) vs Terminated (1)")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Active", "Terminated"])

sat_turnover = df.groupby('EmpSatisfaction')['Termd'].mean()
sns.barplot(x=sat_turnover.index, y=sat_turnover.values, ax=axes[2], palette="coolwarm")
axes[2].set_title("Turnover Rate by Satisfaction Score")
axes[2].set_xlabel("Satisfaction (1 = Low, 5 = High)")
axes[2].set_ylabel("Turnover Rate (%)")

plt.tight_layout()
plt.savefig("hr_insights_eda.png", dpi=150, bbox_inches="tight")
plt.close()
print(" HR Insights graphs saved as 'hr_insights_eda.png' (Department, Salary, Satisfaction)")


 STEP 2.5: HR Business Insights (EDA)
 HR Insights graphs saved as 'hr_insights_eda.png' (Department, Salary, Satisfaction)


In [6]:
print("\n" + "=" * 65)
print("STEP 3: Model Training (XGBoost)")
print("=" * 65)

TARGET = "Termd"
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight,
    eval_metric="logloss", random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(" Model Trained Successfully")
print(f" ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Active", "Terminated"]))

with open("attrition_model.pkl", "wb") as f:
    pickle.dump(model, f)


STEP 3: Model Training (XGBoost)
 Model Trained Successfully
 ROC-AUC Score: 0.7290

Classification Report:
               precision    recall  f1-score   support

      Active       0.79      0.74      0.77        42
  Terminated       0.54      0.62      0.58        21

    accuracy                           0.70        63
   macro avg       0.67      0.68      0.67        63
weighted avg       0.71      0.70      0.70        63



In [7]:
print("\n" + "=" * 65)
print(" STEP 3.5: Performance Visualizations")
print("=" * 65)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Model Evaluation Metrics", fontsize=16, fontweight="bold")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Active", "Terminated"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random Guess")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(loc="lower right")

feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)
feat_imp.plot(kind="barh", ax=axes[2], color="steelblue")
axes[2].set_title("Top 10 Feature Importance (XGBoost internal)")
axes[2].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_performance.png", dpi=150, bbox_inches="tight")
plt.close()
print(" Performance graphs saved as 'model_performance.png' (Confusion Matrix, ROC, Importance)")


 STEP 3.5: Performance Visualizations
 Performance graphs saved as 'model_performance.png' (Confusion Matrix, ROC, Importance)


In [8]:
print("\n" + "=" * 65)
print("STEP 4: Ethical AI Audit")
print("=" * 65)

df_audit = X_test.copy()
df_audit["Predicted_Risk"] = y_pred

if "Sex" in df_audit.columns:
    print("\nDisparate Impact - Gender (Sex):")
    rates_gender = df_audit.groupby("Sex")["Predicted_Risk"].mean()
    for code, rate in rates_gender.items():
        print(f"   Group {code}: {rate:.1%} predicted 'at risk'")

    if len(rates_gender) >= 2:
        di_gender = rates_gender.min() / rates_gender.max()
        print(f"   Disparate Impact Ratio: {di_gender:.3f} (Threshold = 0.8)")
        if di_gender < 0.8:
            print(" BIAS DETECTED: Consider dropping 'Sex' or applying Reweighing.")
        else:
            print(" FAIR: No significant discrimination by gender.")

if "RaceDesc" in df_audit.columns:
    print("\n Disparate Impact - Ethnicity (RaceDesc):")
    rates_eth = df_audit.groupby("RaceDesc")["Predicted_Risk"].mean()
    if len(rates_eth) >= 2:
        di_eth = rates_eth.min() / rates_eth.max()
        print(f"   Disparate Impact Ratio: {di_eth:.3f} (Threshold = 0.8)")


STEP 4: Ethical AI Audit

Disparate Impact - Gender (Sex):
   Group 0: 35.3% predicted 'at risk'
   Group 1: 41.4% predicted 'at risk'
   Disparate Impact Ratio: 0.853 (Threshold = 0.8)
 FAIR: No significant discrimination by gender.

 Disparate Impact - Ethnicity (RaceDesc):
   Disparate Impact Ratio: 0.000 (Threshold = 0.8)


In [9]:
print("\n" + "=" * 65)
print(" STEP 5: Explainable AI (SHAP)")
print("=" * 65)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Global Impact - Employee Turnover Factors", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150)
plt.close()
print(" Global SHAP Plot saved as 'shap_summary.png'")

demo_idx = 0  
sample = X_test.iloc[[demo_idx]]
print(f"\n Individual Explanation for Employee #{demo_idx}:")
print(f"   Risk Probability: {model.predict_proba(sample)[0][1]:.1%}")

shap_local = pd.Series(shap_values[demo_idx], index=X.columns)
top_factors = shap_local.abs().sort_values(ascending=False).head(3)
print("   Key Drivers:")
for feat in top_factors.index:
    direction = "↑ Increases" if shap_local[feat] > 0 else "↓ Decreases"
    print(f"      → {feat} ({sample[feat].values[0]}): {direction} risk (SHAP={shap_local[feat]:+.3f})")

shap_exp = shap.Explanation(values=shap_values[demo_idx], base_values=explainer.expected_value,
                            data=X_test.iloc[demo_idx], feature_names=X.columns.tolist())
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp, show=False)
plt.tight_layout()
plt.savefig("shap_waterfall.png", dpi=150)
plt.close()
print("Local SHAP Plot saved as 'shap_waterfall.png'")


 STEP 5: Explainable AI (SHAP)
 Global SHAP Plot saved as 'shap_summary.png'

 Individual Explanation for Employee #0:
   Risk Probability: 24.7%
   Key Drivers:
      → Salary (62659): ↓ Decreases risk (SHAP=-0.804)
      → RecruitmentSource (1): ↑ Increases risk (SHAP=+0.583)
      → ManagerName (13): ↓ Decreases risk (SHAP=-0.481)
Local SHAP Plot saved as 'shap_waterfall.png'


In [10]:
print("\n" + "=" * 65)
print("EFFICIENT AI — Model training + carbon footprint")
print("=" * 65)

negatives = (y == 0).sum()
positives = (y == 1).sum()
scale_pos_weight = negatives / positives

print(f"Imbalance detected: {negatives} active vs {positives} left")
print(f"scale_pos_weight = {scale_pos_weight:.2f} (auto-adjustment)")

model = XGBClassifier(
    n_estimators=200,         
    max_depth=4,             
    learning_rate=0.05,      
    subsample=0.8,            
    colsample_bytree=0.8,   
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

try:
    from codecarbon import EmissionsTracker

    tracker = EmissionsTracker(log_level="error")
    tracker.start()

    model.fit(X_train, y_train)

    emissions = tracker.stop()

    print(f"\nTraining carbon footprint: {emissions*1000:.4f} g CO₂")
    print(f"   (Equivalent to driving ~{emissions/0.000404:.2f} meters by car)")

except ImportError:
    model.fit(X_train, y_train)

    print("\nCodeCarbon not installed — training without carbon tracking")
    print("   Install it with: pip install codecarbon")

print("XGBoost model trained successfully!")

print("\n Efficient AI comparison (XGBoost vs others):")
print("   XGBoost (our choice) : ~0.01s, ~1MB, CPU only ")
print("   Random Forest        : ~0.05s, ~10MB, CPU only ")
print("   Neural Network       : ~60s,  ~100MB, GPU needed ")


EFFICIENT AI — Model training + carbon footprint
Imbalance detected: 207 active vs 104 left
scale_pos_weight = 1.99 (auto-adjustment)

CodeCarbon not installed — training without carbon tracking
   Install it with: pip install codecarbon
XGBoost model trained successfully!

 Efficient AI comparison (XGBoost vs others):
   XGBoost (our choice) : ~0.01s, ~1MB, CPU only 
   Random Forest        : ~0.05s, ~10MB, CPU only 
   Neural Network       : ~60s,  ~100MB, GPU needed 


In [11]:
!pip install streamlit shap vaderSentiment -q

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import shap
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

st.set_page_config(page_title="HR Guardian AI", page_icon="🛡️", layout="wide")

@st.cache_data
def load_and_prep_data():
    df = pd.read_csv("D:\HackathonGP5\data\processed\cleaned_dataset.csv")

    reference_date = pd.to_datetime('2018-12-31')
    df['DateofHire'] = pd.to_datetime(df['DateofHire'], errors='coerce')
    df['TenureYears'] = (reference_date - df['DateofHire']).dt.days / 365.25
    df['TenureYears'] = df['TenureYears'].fillna(0)

    def generate_feedback(sat):
        if sat >= 4: return "I am very happy here, great team."
        elif sat == 3: return "Things are okay, neutral progression."
        else: return "I am frustrated with my salary."

    analyzer = SentimentIntensityAnalyzer()
    df['Simulated_Review_Notes'] = df['EmpSatisfaction'].apply(generate_feedback)
    df['NLP_Sentiment_Score'] = df['Simulated_Review_Notes'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

    IDENTIFIERS = ["DOB", "State", "Zip", "DateofHire", "LastPerformanceReview_Date", "Simulated_Review_Notes"]
    DATA_LEAKAGE = ["DateofTermination", "TermReason", "EmploymentStatus", "EmpStatusID"]
    df_clean = df.drop(columns=IDENTIFIERS + DATA_LEAKAGE, errors="ignore")

    CATEGORICAL_COLS = ["Sex", "MaritalDesc", "CitizenDesc", "HispanicLatino", "RaceDesc",
                        "Department", "Position", "ManagerName", "RecruitmentSource", "PerformanceScore"]
    le = LabelEncoder()
    for col in CATEGORICAL_COLS:
        if col in df_clean.columns:
            df_clean[col] = le.fit_transform(df_clean[col].astype(str))

    X = df_clean.drop(columns=["Termd"])
    y = df_clean["Termd"]
    return X, y

X, y = load_and_prep_data()

@st.cache_resource
def load_model_and_shap(_X):
    with open("attrition_model.pkl", "rb") as f:
        model = pickle.load(f)

    explainer = shap.TreeExplainer(model)
    shap_values_all = explainer.shap_values(_X)

    probas_all = model.predict_proba(_X)[:, 1]

    return model, shap_values_all, probas_all

model, shap_values_all, probas_all = load_model_and_shap(X)

st.title("HR Anticipa AI dashboard")
st.markdown("*An ethical and efficient AI for employee retention.*")

st.markdown("###  Employee Search")
employee_idx = st.number_input("Enter employee ID (from 0 to 310):", min_value=0, max_value=len(X)-1, value=0, step=1)


real_status = "Left " if y.iloc[employee_idx] == 1 else "Active "
proba_demission = probas_all[employee_idx]
is_at_risk = proba_demission > 0.5

col1, col2, col3 = st.columns(3)
with col1:
    st.metric(label="Actual Status", value=real_status)
with col2:
    st.metric(label="Predicted Risk", value=f"{proba_demission:.1%}")
with col3:
    if is_at_risk:
        st.error("HIGH Risk")
    else:
        st.success("LOW Risk")

st.divider()

st.subheader("Factor Analysis (SHAP)")

col_shap, col_reco = st.columns([2, 1])

with col_shap:
    shap_local = shap_values_all[employee_idx]

    shap_df = pd.DataFrame({'Feature': X.columns, 'Impact': shap_local})
    shap_df['Abs_Impact'] = shap_df['Impact'].abs()
    shap_df = shap_df.sort_values(by='Abs_Impact', ascending=True).tail(5)

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#ff6b6b' if x > 0 else '#4ecdc4' for x in shap_df['Impact']]
    ax.barh(shap_df['Feature'], shap_df['Impact'], color=colors)
    ax.set_title("Top 5 factors influencing this employee", fontweight='bold')
    ax.set_xlabel("<-- Keeps employee | Pushes to leave -->")
    plt.tight_layout()
    st.pyplot(fig, clear_figure=True)

with col_reco:
    st.subheader("HR Actions")
    if not is_at_risk:
        st.info("Employee is stable. Continue current management.")
    else:
        top_risks = shap_df[shap_df['Impact'] > 0].sort_values(by='Impact', ascending=False)['Feature'].tolist()
        if len(top_risks) > 0:
            st.warning("Recommendations:")
            for feature in top_risks:
                if feature == "Salary":
                    st.write("- **Salary**: Review required.")
                elif feature == "EmpSatisfaction" or feature == "NLP_Sentiment_Score":
                    st.write("- **Satisfaction**: Urgent one-on-one meeting.")
                elif feature == "DaysLateLast30" or feature == "Absences":
                    st.write("- **Attendance**: Review workload and engagement.")
                elif feature == "TenureYears":
                    st.write("- **Career growth**: Discuss future opportunities.")
                else:
                    st.write(f"- **{feature}**: Needs further investigation.")
        else:
            st.warning("High overall risk: schedule a meeting.")

Overwriting app.py


In [13]:
pip install streamlit pandas numpy scikit-learn xgboost shap vaderSentiment matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 1. Local tunnel
!npm install localtunnel

import urllib
print("Copy the password / IP :", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py & npx localtunnel --port 8501


up to date, audited 23 packages in 851ms

3 packages are looking for funding
  run `npm fund` for details

2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
Copy the password / IP : 89.30.65.3
